# Phân tích bổ sung — nhóm truy vấn + đường cong dữ liệu (Colab GPU)

Thiết kế và quy tắc đọc kết quả đã chốt **trước khi chạy**: `HANDOFF.md` §7 "Follow-up analyses".

| phần | làm gì | tập |
|---|---|---|
| A | Chia kết quả luồng mô tả tự do theo nhóm truy vấn (có mã lỗi/API/exception vs ngôn ngữ tự nhiên), 4 bước keyword → semantic → hybrid → hybrid+rerank, trước/sau fine-tune | **test** (lần xem thứ hai — chỉ mô tả, không đổi cấu hình) |
| B | Đường cong dữ liệu cho e5: 10% / 25% / 50% / 100% số issue × seed 13, 14 | dev + gold, không test |

**Trước khi chạy**
1. GPU: T4 khoảng 2–2,5 giờ tổng; L4/A100 nhanh hơn nhiều.
2. Cần `followup_bundle.zip` (máy local, thư mục `colab/`). Hai model tải thẳng từ link Google Drive trong ô thứ 4 — link phải để chế độ "Bất kỳ ai có đường liên kết".
3. **Dữ liệu sẽ rời máy sang Google Colab** (như các lần trước).
4. Để tab mở. Ô train bỏ qua model đã train xong nên chạy lại được nếu bị ngắt.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1" datasets accelerate

In [ ]:
from google.colab import files
up = files.upload()            # chon followup_bundle.zip
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline | head -5; ls data/processed

In [ ]:
# Tai 2 model tu Google Drive. Tai khoan Colab co the khac tai khoan so huu file,
# mien la tung file da bat: Share > General access > Anyone with the link > Viewer.
import os, glob, shutil, zipfile
FILES = {"e5-base-task2": "1rNO17nLa1kXGdmMEsUhW5Hl_9e7kjN8O",
         "bge-reranker-base-task": "12x-H0yLkedkl0_94K_CHV621cZuncRqn"}
!pip -q install -U gdown
import gdown

os.makedirs("models", exist_ok=True)
for name, fid in FILES.items():
    dst = "models/" + name
    if os.path.exists(dst + "/config.json"):
        print("da co", dst); continue

    z = "/content/" + name + ".zip"
    # Xoa file tai do dang/HTML loi tu lan chay truoc.
    if os.path.exists(z) and not zipfile.is_zipfile(z):
        os.remove(z)
    if not os.path.exists(z):
        result = gdown.download(id=fid, output=z, quiet=False)
        assert result and os.path.exists(z), (
            "Khong tai duoc file Drive. Hay mo link bang cua so an danh va kiem tra "
            "General access = Anyone with the link (Viewer). File ID: " + fid
        )
    assert zipfile.is_zipfile(z), (
        z + " khong phai ZIP; co the file chua public hoac vuot gioi han tai cua Drive."
    )

    tmp = "/content/unz_" + name
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(z) as archive:
        archive.extractall(tmp)
    # Thu muc model = config.json nong nhat (e5 con co 1_Pooling/config.json).
    cfg = sorted(glob.glob(tmp + "/**/config.json", recursive=True), key=len)
    assert cfg, "khong thay config.json trong " + z + " -- file ZIP co dung la model?"
    shutil.rmtree(dst, ignore_errors=True)
    shutil.move(os.path.dirname(cfg[0]), dst)
    print(name, "->", dst)
for m in FILES:
    print("models/" + m, os.path.exists("models/" + m + "/config.json"))

## 0. Chạy thử (embedding sinh ra ở đây được cache, các bước sau dùng lại)

In [ ]:
!python pipeline/train_task_models.py --model bi --pairs data/processed/task_pairs_v2.jsonl --fraction 0.1 --out /tmp/lc_smoke --max-steps 3 2>&1 | tail -2
!python pipeline/group_analysis_text.py --device cuda --limit 10 2>&1 | tail -15
!python pipeline/learning_curve_eval.py --device cuda --limit 10 --ref f100-s13 --models f000=intfloat/multilingual-e5-base f100-s13=models/e5-base-task2 2>&1 | tail -8

## A. Nhóm truy vấn — luồng mô tả tự do, test (chạy MỘT lần, ~20–25 phút T4)

In [ ]:
!python pipeline/group_analysis_text.py --device cuda 2>&1 | grep -v "Batches" | tail -150

## B1. Train 7 model cho đường cong (~40 phút T4). 100% / seed 13 = `models/e5-base-task2` có sẵn

In [ ]:
RUNS = [(0.1, 13), (0.1, 14), (0.25, 13), (0.25, 14), (0.5, 13), (0.5, 14), (1.0, 14)]
def lc_name(f, s):
    return "e5-f%03d-s%d" % (round(f * 100), s)
for f, s in RUNS:
    out = "models/lc/" + lc_name(f, s)
    if os.path.exists(out + "/model.safetensors"):
        print("da co", out); continue
    !python pipeline/train_task_models.py --model bi --pairs data/processed/task_pairs_v2.jsonl --fraction {f} --seed {s} --out {out} 2>&1 | tail -1
!ls models/lc

## B2. Đo đường cong trên dev + gold (~60 phút T4)

In [ ]:
M = ["f000=intfloat/multilingual-e5-base", "f100-s13=models/e5-base-task2"]
M += [lc_name(f, s)[3:] + "=models/lc/" + lc_name(f, s) for f, s in RUNS]
MODELS_ARG = " ".join(M)
print(MODELS_ARG)
!python pipeline/learning_curve_eval.py --device cuda --ref f100-s13 --models {MODELS_ARG} 2>&1 | grep -v "Batches" | tail -20

## Tải kết quả (file nhỏ; không cần tải model)

In [ ]:
!zip -q followup_results.zip data/processed/group_text_test.json data/processed/group_text_test_ranks.jsonl data/processed/learning_curve_dev.json models/lc/*/task_meta.json
if os.path.isdir("/content/drive/MyDrive"):
    !cp followup_results.zip /content/drive/MyDrive/ && echo "da luu MyDrive/followup_results.zip"
files.download("followup_results.zip")